# Phase 2.2: Feature Quality Checks
**DNA Gene Mapping Project - ML Phase**  
**Author:** Sharique Mohammad  
**Date:** February 2026

## Objective
Identify and remove low-quality features before modeling

## Key Tasks
1. Detect high missing value features (>80%)
2. Identify zero-variance features
3. Find duplicated columns
4. Validate data types
5. Flag suspicious patterns

## Deliverables
- Feature quality report
- List of features to remove
- Clean feature set for correlation analysis

## Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from pathlib import Path
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

PROJECT_ROOT = Path().absolute().parent.parent
REPORTS_DIR = PROJECT_ROOT / 'data' / 'analytical' / 'reports'
FIGURES_DIR = PROJECT_ROOT / 'data' / 'analytical' / 'figures' / 'feature_quality'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

print("Setup complete")
print(f"Reports: {REPORTS_DIR}")
print(f"Figures: {FIGURES_DIR}")

In [ ]:
# Database connection
load_dotenv()

POSTGRES_HOST = os.getenv('POSTGRES_HOST', 'localhost')
POSTGRES_PORT = os.getenv('POSTGRES_PORT', '5432')
POSTGRES_DB = os.getenv('POSTGRES_DB', 'genome_db')
POSTGRES_USER = os.getenv('POSTGRES_USER', 'postgres')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)

print("Database connection established")

## 1. Load Data for Quality Checks

In [ ]:
# Load all 5 feature tables with sampling for quality checks
print("Loading feature tables for quality analysis...")

tables = {
    'clinical_ml_features': 'TABLESAMPLE SYSTEM (5)',
    'disease_ml_features': 'TABLESAMPLE SYSTEM (5)',
    'pharmacogene_ml_features': 'TABLESAMPLE SYSTEM (5)',
    'variant_impact_ml_features': 'TABLESAMPLE SYSTEM (5)',
    'structural_variant_ml_features': 'TABLESAMPLE SYSTEM (10)'
}

dataframes = {}

for table_name, sample_clause in tables.items():
    query = f"SELECT * FROM gold.{table_name} {sample_clause}"
    df = pd.read_sql(query, engine)
    dataframes[table_name] = df
    print(f"  {table_name}: {len(df):,} rows, {len(df.columns)} columns")

print(f"\nLoaded {len(dataframes)} tables for quality analysis")

## 2. Missing Value Analysis

In [ ]:
# Analyze missing values for each table
high_missing_features = {}
MISSING_THRESHOLD = 0.80

print(f"Identifying features with >{MISSING_THRESHOLD*100:.0f}% missing values...\n")

for table_name, df in dataframes.items():
    missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    high_missing = missing_pct[missing_pct > MISSING_THRESHOLD * 100]
    
    if len(high_missing) > 0:
        high_missing_features[table_name] = high_missing.to_dict()
        print(f"{table_name}:")
        print(f"  Features with >{MISSING_THRESHOLD*100:.0f}% missing: {len(high_missing)}")
        for col, pct in high_missing.items():
            print(f"    {col}: {pct:.2f}%")
        print()
    else:
        print(f"{table_name}: No features with >{MISSING_THRESHOLD*100:.0f}% missing\n")

total_high_missing = sum(len(v) for v in high_missing_features.values())
print(f"Total features to remove (high missing): {total_high_missing}")

In [ ]:
# Visualize missing value distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (table_name, df) in enumerate(dataframes.items()):
    missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    
    axes[idx].barh(range(min(20, len(missing_pct))), 
                   missing_pct.head(20).values,
                   color='coral', alpha=0.7, edgecolor='black')
    axes[idx].axvline(MISSING_THRESHOLD * 100, color='red', linestyle='--', 
                      linewidth=2, label=f'>{MISSING_THRESHOLD*100:.0f}% threshold')
    axes[idx].set_xlabel('Missing %', fontsize=10, fontweight='bold')
    axes[idx].set_title(table_name.replace('_ml_features', ''), fontsize=11, fontweight='bold')
    axes[idx].set_yticks([])
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

if len(dataframes) < 6:
    fig.delaxes(axes[5])

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_missing_values_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {FIGURES_DIR / '01_missing_values_distribution.png'}")

## 3. Zero-Variance Features

In [ ]:
# Identify features with zero or near-zero variance
zero_variance_features = {}

print("Identifying zero-variance features...\n")

for table_name, df in dataframes.items():
    zero_var = []
    
    for col in df.columns:
        if col in ['variant_id', 'sv_id', 'gene_name']:
            continue
        
        nunique = df[col].nunique()
        
        if nunique == 1:
            zero_var.append(col)
    
    if zero_var:
        zero_variance_features[table_name] = zero_var
        print(f"{table_name}:")
        print(f"  Zero-variance features: {len(zero_var)}")
        for col in zero_var:
            print(f"    {col}")
        print()
    else:
        print(f"{table_name}: No zero-variance features\n")

total_zero_var = sum(len(v) for v in zero_variance_features.values())
print(f"Total features to remove (zero variance): {total_zero_var}")

## 4. Duplicated Columns

In [ ]:
# Detect duplicated columns (same values)
duplicated_features = {}

print("Identifying duplicated columns...\n")

for table_name, df in dataframes.items():
    duplicates = []
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    checked = set()
    for i, col1 in enumerate(numeric_cols):
        if col1 in checked:
            continue
        
        for col2 in numeric_cols[i+1:]:
            if col2 in checked:
                continue
            
            if df[col1].equals(df[col2]):
                duplicates.append((col1, col2))
                checked.add(col2)
    
    if duplicates:
        duplicated_features[table_name] = duplicates
        print(f"{table_name}:")
        print(f"  Duplicate column pairs: {len(duplicates)}")
        for col1, col2 in duplicates:
            print(f"    {col1} = {col2}")
        print()
    else:
        print(f"{table_name}: No duplicated columns\n")

total_duplicates = sum(len(v) for v in duplicated_features.values())
print(f"Total duplicate column pairs: {total_duplicates}")

## 5. Data Type Validation

In [ ]:
# Check if data types are correct after PostgreSQL conversion
print("Validating data types...\n")

type_issues = {}

for table_name, df in dataframes.items():
    issues = []
    
    for col in df.columns:
        dtype = df[col].dtype
        
        # Check if boolean columns are properly typed
        if col.startswith('is_') or col.startswith('has_') or col.startswith('target_') or col.startswith('affects_'):
            if dtype != 'bool':
                issues.append(f"{col}: Expected bool, got {dtype}")
        
        # Check if score/count columns are numeric
        if 'score' in col or 'count' in col or 'ratio' in col:
            if dtype == 'object':
                issues.append(f"{col}: Expected numeric, got object")
    
    if issues:
        type_issues[table_name] = issues
        print(f"{table_name}:")
        print(f"  Type issues: {len(issues)}")
        for issue in issues[:10]:
            print(f"    {issue}")
        if len(issues) > 10:
            print(f"    ... and {len(issues)-10} more")
        print()
    else:
        print(f"{table_name}: All data types correct\n")

total_type_issues = sum(len(v) for v in type_issues.values())

if total_type_issues > 0:
    print(f"\nWARNING: {total_type_issues} type issues detected!")
    print("Action: PostgreSQL type conversion script may need re-run")
else:
    print("\nPASS: All data types are correct")

## 6. Feature Cardinality Check

In [ ]:
# Check cardinality of categorical features
print("Analyzing categorical feature cardinality...\n")

high_cardinality_features = {}
HIGH_CARDINALITY_THRESHOLD = 1000

for table_name, df in dataframes.items():
    high_card = []
    
    object_cols = df.select_dtypes(include=['object']).columns
    
    for col in object_cols:
        if col in ['variant_id', 'sv_id']:
            continue
        
        nunique = df[col].nunique()
        
        if nunique > HIGH_CARDINALITY_THRESHOLD:
            high_card.append((col, nunique))
    
    if high_card:
        high_cardinality_features[table_name] = high_card
        print(f"{table_name}:")
        print(f"  High cardinality features (>{HIGH_CARDINALITY_THRESHOLD:,} unique): {len(high_card)}")
        for col, nunique in high_card:
            print(f"    {col}: {nunique:,} unique values")
        print()
    else:
        print(f"{table_name}: No high cardinality features\n")

print("Note: High cardinality features may need special encoding or removal")

## 7. Summary Statistics

In [ ]:
# Generate summary of feature quality issues
summary_data = []

for table_name in dataframes.keys():
    total_features = len(dataframes[table_name].columns)
    high_missing = len(high_missing_features.get(table_name, []))
    zero_var = len(zero_variance_features.get(table_name, []))
    duplicates = len(duplicated_features.get(table_name, []))
    type_issues_count = len(type_issues.get(table_name, []))
    
    to_remove = high_missing + zero_var + duplicates
    clean_features = total_features - to_remove
    
    summary_data.append({
        'Table': table_name.replace('_ml_features', ''),
        'Total Features': total_features,
        'High Missing': high_missing,
        'Zero Variance': zero_var,
        'Duplicates': duplicates,
        'Type Issues': type_issues_count,
        'To Remove': to_remove,
        'Clean Features': clean_features
    })

summary_df = pd.DataFrame(summary_data)

print("\nFeature Quality Summary:")
print("="*100)
print(summary_df.to_string(index=False))
print("="*100)

print(f"\nOverall Statistics:")
print(f"  Total features across all tables: {summary_df['Total Features'].sum()}")
print(f"  Features to remove: {summary_df['To Remove'].sum()}")
print(f"  Clean features remaining: {summary_df['Clean Features'].sum()}")
print(f"  Retention rate: {summary_df['Clean Features'].sum()/summary_df['Total Features'].sum()*100:.1f}%")

In [ ]:
# Visualize feature quality summary
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(summary_df))
width = 0.2

ax.bar(x - width*1.5, summary_df['High Missing'], width, label='High Missing', color='#e74c3c', alpha=0.7)
ax.bar(x - width*0.5, summary_df['Zero Variance'], width, label='Zero Variance', color='#f39c12', alpha=0.7)
ax.bar(x + width*0.5, summary_df['Duplicates'], width, label='Duplicates', color='#9b59b6', alpha=0.7)
ax.bar(x + width*1.5, summary_df['Type Issues'], width, label='Type Issues', color='#3498db', alpha=0.7)

ax.set_xlabel('Table', fontsize=11, fontweight='bold')
ax.set_ylabel('Feature Count', fontsize=11, fontweight='bold')
ax.set_title('Feature Quality Issues by Table', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(summary_df['Table'], rotation=15)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_quality_issues_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {FIGURES_DIR / '02_quality_issues_summary.png'}")

## 8. Generate Feature Quality Report

In [ ]:
# Generate comprehensive feature quality report
report_path = REPORTS_DIR / 'feature_quality_report.txt'

with open(report_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("FEATURE QUALITY CHECKS REPORT\n")
    f.write("DNA Gene Mapping Project - Phase 2.2\n")
    f.write("="*80 + "\n\n")
    
    f.write("SUMMARY\n")
    f.write("-"*80 + "\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n\n")
    
    f.write("OVERALL STATISTICS\n")
    f.write("-"*80 + "\n")
    f.write(f"Total features: {summary_df['Total Features'].sum()}\n")
    f.write(f"Features to remove: {summary_df['To Remove'].sum()}\n")
    f.write(f"Clean features: {summary_df['Clean Features'].sum()}\n")
    f.write(f"Retention rate: {summary_df['Clean Features'].sum()/summary_df['Total Features'].sum()*100:.1f}%\n\n")
    
    f.write("DETAILED FINDINGS\n")
    f.write("="*80 + "\n\n")
    
    if high_missing_features:
        f.write("1. HIGH MISSING VALUE FEATURES (>80%)\n")
        f.write("-"*80 + "\n")
        for table, features in high_missing_features.items():
            f.write(f"\n{table}:\n")
            for col, pct in features.items():
                f.write(f"  - {col}: {pct:.2f}%\n")
        f.write("\n")
    
    if zero_variance_features:
        f.write("2. ZERO VARIANCE FEATURES\n")
        f.write("-"*80 + "\n")
        for table, features in zero_variance_features.items():
            f.write(f"\n{table}:\n")
            for col in features:
                f.write(f"  - {col}\n")
        f.write("\n")
    
    if duplicated_features:
        f.write("3. DUPLICATED COLUMNS\n")
        f.write("-"*80 + "\n")
        for table, pairs in duplicated_features.items():
            f.write(f"\n{table}:\n")
            for col1, col2 in pairs:
                f.write(f"  - {col1} = {col2}\n")
        f.write("\n")
    
    if type_issues:
        f.write("4. DATA TYPE ISSUES\n")
        f.write("-"*80 + "\n")
        for table, issues in type_issues.items():
            f.write(f"\n{table}: {len(issues)} issues\n")
            for issue in issues[:20]:
                f.write(f"  - {issue}\n")
            if len(issues) > 20:
                f.write(f"  ... and {len(issues)-20} more\n")
        f.write("\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("RECOMMENDATIONS\n")
    f.write("="*80 + "\n")
    f.write("1. Remove all high missing value features (>80%)\n")
    f.write("2. Remove all zero-variance features\n")
    f.write("3. Keep only one from each duplicated pair\n")
    if type_issues:
        f.write("4. Re-run PostgreSQL type conversion for tables with type issues\n")
    f.write("\n")
    f.write("NEXT STEPS\n")
    f.write("-"*80 + "\n")
    f.write("- Proceed to Phase 2.3: Correlation Analysis\n")
    f.write("- Identify highly correlated features (r > 0.95)\n")
    f.write("- Remove redundant features\n")

print(f"\nReport saved: {report_path}")
print("\n" + "="*80)
print("PHASE 2.2 COMPLETE - Feature Quality Checks")
print("="*80)
print(f"\nGenerated {len(list(FIGURES_DIR.glob('*.png')))} visualizations")
print(f"Reports: {REPORTS_DIR}")
print("\nQuality Issues Identified:")
print(f"  High missing: {total_high_missing}")
print(f"  Zero variance: {total_zero_var}")
print(f"  Duplicates: {total_duplicates}")
print(f"  Type issues: {total_type_issues}")
print(f"\nTotal features to remove: {summary_df['To Remove'].sum()}")
print("\nNext: Phase 2.3 - Correlation Analysis")